In [0]:
employee_df=spark.read.format("csv").option("inferschema" , "True").option("header", "True").load("abfss://devcontainer@discoveryprodfixp.dfs.core.windows.net/staging-target/employees (1).csv")

employee_df.display()


In [0]:
departments_df=spark.read.format("csv").option("inferschema" , "True").option("header", "True").load("abfss://devcontainer@discoveryprodfixp.dfs.core.windows.net/staging-target/departments.csv")

departments_df.display()

In [0]:

employee_df.write.format("delta").mode("overwrite").saveAsTable("shivasai_catalog.bronze.employee_stage")

In [0]:
departments_df.write.format("delta").mode("overwrite").saveAsTable("shivasai_catalog.bronze.departments_stage")

In [0]:
%sql
create schema if not exists shivasai_catalog.silver_schema;
CREATE OR REPLACE TABLE shivasai_catalog.silver_schema.employee_target
USING DELTA
AS
SELECT
    e.emp_id,
    e.emp_name,
    e.dept_id,
    e.salary,
    e.salary * 0.10 AS bonus,
    CASE
        WHEN e.salary >= 40000 THEN 'High'
        WHEN e.salary >= 30000 THEN 'Medium'
        ELSE 'Low'
    END AS salary_category,
    d.dept_name
FROM shivasai_catalog.bronze_schema.employee_stage e
LEFT JOIN shivasai_catalog.bronze_schema.departments_stage d
    ON e.dept_id = d.dept_id;

In [0]:
%sql
select * from shivasai_catalog.silver_schema.employee_target